# 03 — Modelo Principal: Bagging — Random Forest
Predicción Won (1) vs Lost (0) — CRM Sales Opportunities

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, roc_auc_score, RocCurveDisplay
)
from paths import PROCESSED_DIR

X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    max_features="sqrt",
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 macro : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"AUC-ROC  : {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred, target_names=["Lost", "Won"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Lost", "Won"], yticklabels=["Lost", "Won"])
plt.title("Matriz de confusión — Random Forest (CRM)")
plt.ylabel("Real"); plt.xlabel("Predicción")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Curva ROC — Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=X_train.columns)
importance.sort_values(ascending=True).tail(15).plot(kind="barh", figsize=(8, 6))
plt.title("Top 15 — Importancia de variables (Random Forest)")
plt.xlabel("Importancia")
plt.tight_layout()
plt.show()

joblib.dump(rf_model, PROCESSED_DIR / "rf_model.joblib")
print("Modelo guardado en:", PROCESSED_DIR / "rf_model.joblib")